# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list the available record sets in this Croissant schema and display each one's field names and `@id`s. Record set and field `@id` values should be used for all further reference.

In [ ]:
from pprint import pprint
record_sets = []

if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set
else:
    print("No record sets found in the metadata.")
    record_sets = []

if not record_sets:
    print("No record sets explicitly defined in the dataset metadata. Attempting to auto-discover any available RecordSet.")
    # Try to infer typical structure
    from mlcroissant.utils import find_record_sets
    found = find_record_sets(metadata)
    if found:
        record_sets = found

# Get each record set's @id and field @ids
record_set_info = {}
record_set_ids = []

for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, 'id', None)
    record_set_ids.append(rs_id)
    # Try to obtain fields
    fields = []
    if isinstance(rs, dict):
        fields = rs.get('field', [])
    elif hasattr(rs, 'field'):
        fields = rs.field

    # Build list of field IDs and names (if available)
    field_list = []
    for f in fields:
        if isinstance(f, dict):
            fid = f.get('@id', None)
            fname = f.get('name', fid)
        else:
            fid = getattr(f, 'id', None)
            fname = getattr(f, 'name', fid)
        field_list.append({'@id': fid, 'name': fname})
    record_set_info[rs_id] = field_list

if not record_set_ids:
    print("No record sets available in this dataset.")
else:
    print("Record sets available in the dataset (by @id):")
    pprint(record_set_ids)
    # Show first record set's fields
    sample_rs = record_set_ids[0]
    print(f"\nFields in record set '{sample_rs}':")
    pprint(record_set_info[sample_rs])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

*Note: The FAIR² Croissant schema may only have a single principal record set (often named for the main table or tabular resource).*

In [ ]:
# If you know the correct record set ID, put it here.
if record_set_ids:
    selected_record_sets = record_set_ids  # Use all discovered
else:
    selected_record_sets = []

dataframes = {}
for rsid in selected_record_sets:
    print(f"Loading records for record set {rsid} ...")
    try:
        # By convention, pass `record_set=...` as the @id
        recs = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(recs)
        dataframes[rsid] = df
        print(f"Records loaded: {len(df)}")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for record set {rsid}: {e}")

# Choose the primary dataframe for analysis (if only one available)
if selected_record_sets:
    main_record_set_id = selected_record_sets[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section demonstrates: removing outliers, transforming numeric data, and grouping by a key attribute.

In [ ]:
# EDA: Filtering, Normalization, Grouping
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Try to guess a numeric field (commonly: 'Coverage (%)', 'Peptide Count', 'MW', etc.)
    possible_numeric_fields = [col for col in df.columns if ('coverage' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower() or df[col].dtype in ['int64','float64'])]
    print("Available numeric-like fields:", possible_numeric_fields)
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
    else:
        print("No numeric field automatically detected.")
        numeric_field = None

    # Try a typical filtering threshold
    threshold = 10
    if numeric_field:
        try:
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} row(s)")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a categorical field (e.g. 'Description' or similar)
            possible_groups = [col for col in df.columns if col.lower() in ['description', 'accession', 'protein', 'group', 'type', 'annotation'] or df[col].dtype == object]
            group_field = None
            for g in ['description','accession','group','type']:
                if g in [c.lower() for c in df.columns]:
                    group_field = [c for c in df.columns if c.lower()==g][0]
                    break
            if not group_field and possible_groups:
                group_field = possible_groups[0]

            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped filtered data by '{group_field}':")
                display(grouped_df.head())
            else:
                print("No suitable categorical field found for grouping.")
        except Exception as e:
            print(f"Error during EDA: {e}")
    else:
        print("Cannot proceed with EDA due to missing numeric field.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will create a histogram for our selected numeric field, and a simple boxplot if multiple entries are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        group_counts = df[group_field].value_counts().sort_values(ascending=False).head(10).index
        sub = df[df[group_field].isin(group_counts)]
        sns.boxplot(data=sub, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (Top 10 categories)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect a Croissant-described FAIR² dataset with `mlcroissant`,
- Enumerate record sets and their fields by `@id`,
- Extract tabular data into DataFrames,
- Perform basic exploratory data analysis (filtering, normalization, group-wise summarization), and
- Visualize data distributions and relationships.

This methodology facilitates auditable, FAIR-compliant workflows for complex tabular scientific datasets.